In [1]:
# Install packages
# %pip install -r requirements.txt

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import math

In [3]:
def code_break():
    print("-----------------------------------------")

def get_ssr(matrix: np.ndarray, estimators: np.array):
    ""

## Estimators

In [4]:
apple = pd.read_csv("APPLE.csv")
apple.head()

,id,educ,date,state,regprc,ecoprc,inseason,hhsize,male,faminc,age,reglbs,ecolbs,numlt5,num5_17,num18_64,numgt64
0,10002,16,111597,SD,1.19,1.19,1,4,0,45,43,2.0,2.000000,0,1,3,0
1,10004,16,121897,KS,0.59,0.79,0,1,0,65,37,0.0,2.000000,0,0,1,0
2,10034,18,111097,MI,0.59,0.99,1,3,0,65,44,0.0,2.666667,0,2,1,0
3,10035,12,111597,TN,0.89,1.09,1,2,1,55,55,3.0,0.000000,0,0,2,0
4,10039,15,122997,NY,0.89,1.09,0,1,1,25,22,0.0,3.000000,0,0,1,0


In [18]:
X = apple[['regprc', 'ecoprc', 'inseason']]
X = sm.add_constant(X)
y = apple['ecolbs']

# ----- Manual Calc ------
X = np.array(X)
XtX = X.T @ X
det_XtX = np.linalg.det(XtX)
display(f"Determinant XtX: {det_XtX}")
inv_XtX = np.linalg.inv(XtX)
print(inv_XtX)

code_break()

Xty = X.T @ y
print(Xty)

code_break()

estimators = inv_XtX @ Xty
print(estimators)

estimators_results = zip(['const', 'x1', 'x2', 'x3'], estimators)
estimators_results = pd.DataFrame(estimators_results)

estimators_results

'Determinant XtX: 68134998.54239926'

[[ 2.46539584e-02 -1.14262776e-02 -1.11706080e-02 -2.88776250e-03]
 [-1.14262776e-02  8.19466817e-02 -5.63098238e-02 -3.08169376e-05]
 [-1.11706080e-02 -5.63098238e-02  5.61084858e-02  5.78681982e-04]
 [-2.88776250e-03 -3.08169376e-05  5.78681982e-04  6.80546847e-03]]
-----------------------------------------
[ 972.8333302   862.27166293 1003.47166213  308.1666625 ]
-----------------------------------------
[ 2.03233643  3.02962864 -2.9398991  -0.15797471]


,0,1
0,const,2.032336
1,x1,3.029629
2,x2,-2.939899
3,x3,-0.157975


In [6]:
model1 = sm.OLS(exog=X, endog=y)
results = model1.fit()

results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 ecolbs   R-squared:                       0.037
Model:                            OLS   Adj. R-squared:                  0.033
Method:                 Least Squares   F-statistic:                     8.469
Date:                Tue, 17 Nov 2026   Prob (F-statistic):           1.58e-05
Time:                        15:28:44   Log-Likelihood:                -1535.0
No. Observations:                 660   AIC:                             3078.
Df Residuals:                     656   BIC:                             3096.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.0323      0.390      5.211      0.000       1.267       2.798
x1             3.0296      0.711      4.261      0.000       1.633       4.426
x2            -2.9399      0.588     -4.997      0.000      -4.095      -1.785
x3            -0.1580      0.205     -0.771      0.441      -0.560       0.244
==============================================================================
Omnibus:                      989.100   Durbin-Watson:                   2.025
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           351898.328
Skew:                           8.279   Prob(JB):                         0.00
Kurtosis:                     114.902   Cond. No.                         16.3
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [ ]:
stats_package_est = pd.DataFrame(results.params)
# comparison_table.reset_index(inplace=True)
# stats_package_est.rename(mapper={0:'stats_package'}, inplace=True, axis=1)
stats_package_est


# comparison_table.index.name = None

# display(comparison_table)

,0
const,2.032336
x1,3.029629
x2,-2.939899
x3,-0.157975


## Standard Errors

In [ ]:
# Manual Calculation
beta = np.asarray(estimators).reshape(-1)
e = np.asarray(y).reshape(-1) - X @ beta
e_2 = e**2
ssr = e_2.sum()

s_2 = ssr / (X.shape[0] - X.shape[1])

cov_matrix = s_2*(inv_XtX)
print(cov_matrix)

In [ ]:
manual_cov_matrix = pd.DataFrame(
    cov_matrix, 
    index=['const', 'regprc', 'ecoprc', 'inseason'],
    columns=['const', 'regprc', 'ecoprc', 'inseason']
)
display(manual_cov_matrix)

In [ ]:
sp_cov_matrix = results.cov_params()
for i in range(0, 2):
    sp_cov_matrix.rename(mapper={
        'const':'const',
        'x1':'regprc',
        'x2':'ecoprc',
        'x3':'inseason',
    },
    axis=i,
    inplace=True
    )

sp_cov_matrix

## Model 2

In [ ]:
X = apple[['ecoprc', 'regprc', 'reglbs', 'age', 'faminc', 'hhsize', 'numlt5']]
X = sm.add_constant(X)
y = apple['ecolbs']

In [ ]:
model2 = sm.OLS(exog=X, endog=y)
results = model2.fit()

results.summary()

## Test, ecolbs >= 0, ecolbs < 0

## HC2 Errors

In [ ]:
## Diagonal
X_np = np.asarray(X)
inv_XtX = np.linalg.inv(X_np.T @ X_np)
projection_matrix = X_np @ inv_XtX @ X_np.T
# print(projection_matrix)
# code_break()
# print(projection_matrix.shape)
# code_break()
px_diag = projection_matrix.diagonal()

## residuals
u2 = np.asarray((results.resid)**2)

print(type(px_diag),type(u2))
print(px_diag.shape, u2.shape)

## Omega
omega_weights = np.asarray(results.resid)**2 / (1 - px_diag)

## HC2 Errors
XtX = X.T @ X
inv_XtX = np.linalg.inv(XtX)

hc2_cov = inv_XtX @ (X_np.T * omega_weights) @ X_np @ inv_XtX
code_break()
pd.DataFrame(hc2_cov, columns=model2.exog_names, index=model2.exog_names)

<class 'numpy.ndarray'> <class 'numpy.ndarray'>
(660,) (660,)
-----------------------------------------


,const,ecoprc,regprc,reglbs,age,faminc,hhsize,numlt5
const,0.212772,-0.136302,0.156867,-2.536973e-03,-0.002762,-2.742463e-04,-0.013898,-0.017852
ecoprc,-0.136302,0.317567,-0.465830,-3.140758e-03,0.002669,5.515218e-05,0.017157,0.010413
regprc,0.156867,-0.465830,0.844100,3.079773e-03,-0.005506,2.477389e-04,-0.037203,-0.020476
reglbs,-0.002537,-0.003141,0.003080,2.087790e-03,0.000018,-5.829303e-07,-0.000229,0.000625
age,-0.002762,0.002669,-0.005506,1.790747e-05,0.000076,-2.644274e-06,0.000319,0.000489
faminc,-0.000274,0.000055,0.000248,-5.829303e-07,-0.000003,4.671345e-06,-0.000034,-0.000005
hhsize,-0.013898,0.017157,-0.037203,-2.288426e-04,0.000319,-3.394481e-05,0.005283,-0.003016
numlt5,-0.017852,0.010413,-0.020476,6.245691e-04,0.000489,-4.607283e-06,-0.003016,0.022393


In [ ]:
pd.DataFrame(results.cov_HC2, columns=model2.exog_names, index=model2.exog_names)

,const,ecoprc,regprc,reglbs,age,faminc,hhsize,numlt5
const,0.212772,-0.136302,0.156867,-2.536973e-03,-0.002762,-2.742463e-04,-0.013898,-0.017852
ecoprc,-0.136302,0.317567,-0.465830,-3.140758e-03,0.002669,5.515218e-05,0.017157,0.010413
regprc,0.156867,-0.465830,0.844100,3.079773e-03,-0.005506,2.477389e-04,-0.037203,-0.020476
reglbs,-0.002537,-0.003141,0.003080,2.087790e-03,0.000018,-5.829303e-07,-0.000229,0.000625
age,-0.002762,0.002669,-0.005506,1.790747e-05,0.000076,-2.644274e-06,0.000319,0.000489
faminc,-0.000274,0.000055,0.000248,-5.829303e-07,-0.000003,4.671345e-06,-0.000034,-0.000005
hhsize,-0.013898,0.017157,-0.037203,-2.288426e-04,0.000319,-3.394481e-05,0.005283,-0.003016
numlt5,-0.017852,0.010413,-0.020476,6.245691e-04,0.000489,-4.607283e-06,-0.003016,0.022393
